In [ ]:
import numpy as np
from music21 import converter, instrument, note, chord
import tensorflow as tf

In [ ]:
import glob

midi_files = glob.glob("dataset/midi_dataset/*.mid")

print("Total MIDI Files:", len(midi_files))
print(midi_files[:5])

1. Create an empty list to store all the musical notes extracted from every MIDI file.
2. Open each MIDI file one by one from the dataset.
3. Read the music information stored inside each MIDI file.
4. Check whether the MIDI file contains separate instrument tracks.
5. If separate instrument tracks exist, select the first instrument for processing.
6. If there are no separate tracks, use all the notes available in the MIDI file.
7. Go through every musical element one by one.
8. If the element is a single note, store its pitch name in the list.
9. If the element is a chord, store all the notes of that chord together as one pattern.
10. Repeat the process for every MIDI file, ignore files with errors, and finally display the total number of extracted notes and the first few notes.


In [ ]:
notes = []

for file in midi_files:
    try:
        midi = converter.parse(file)

        parts = instrument.partitionByInstrument(midi)

        if parts:
            notes_to_parse = parts.parts[0].recurse()
        else:
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))

    except Exception as e:
        print(f"Error reading {file}: {e}")

print("Total Notes:", len(notes))
print(notes[:20])

1. Create a sorted list of all unique notes and chords by removing duplicate values from the extracted notes.
2. Display the total number of unique note and chord patterns that will be used to train the deep learning model.


In [ ]:
pitchnames = sorted(set(notes))

print("Total Unique Notes:", len(pitchnames))

1. Assign a unique integer number to every unique note and chord so the deep learning model can understand them as numerical values.
2. Display the total vocabulary size, which represents the number of unique notes and chord patterns available for training the model.


In [ ]:

note_to_int = {note: number for number, note in enumerate(pitchnames)}

print("Vocabulary Size:", len(note_to_int))



1. Set the input sequence length to 100, meaning the model will learn from the previous 100 notes to predict the next note.

2. Create two empty lists to store the input sequences and their corresponding target outputs.

3. Move through the entire note dataset using a sliding window of 100 notes.

4. For each window, use the first 100 notes as the input sequence and the 101st note as the expected output.

5. Convert all notes and chords into their corresponding integer values using the note-to-integer mapping and store them in the input and output lists.

6. Display the total number of input sequences and output sequences created for training the deep learning model.


In [ ]:
sequence_length = 100

network_input = []
network_output = []

for i in range(len(notes) - sequence_length):
    sequence_in = notes[i:i + sequence_length]
    sequence_out = notes[i + sequence_length]

    network_input.append([note_to_int[n] for n in sequence_in])
    network_output.append(note_to_int[sequence_out])

print("Input Sequences:", len(network_input))
print("Output Sequences:", len(network_output))

1. Calculate the total number of training patterns created from the input sequences.

2. Reshape the input data into a 3D format `(samples, time steps, features)` so it can be used by the LSTM model.

3. Normalize the input values by dividing them by the total number of unique notes, bringing all values into a smaller range between 0 and 1.

4. Display the final shape of the input data to verify that it is correctly prepared for training the deep learning model.


In [ ]:

n_patterns = len(network_input)

network_input = np.reshape(
    network_input,
    (n_patterns, sequence_length, 1)
)

network_input = network_input / float(len(pitchnames))

print(network_input.shape)



1. Convert the output note numbers into one-hot encoded vectors so the model can classify and predict the next note correctly.

2. Display the shape of the encoded output data to verify that it is ready for training the deep learning model.


In [ ]:

network_output = tf.keras.utils.to_categorical(network_output)

print(network_output.shape)



1. Create a Sequential deep learning model to build the music generation network layer by layer.

2. Add three LSTM layers with 512 neurons each to learn musical patterns and long-term relationships between notes.

3. Add Dropout layers after the LSTM layers to reduce overfitting and improve the model's ability to generalize.

4. Add a Dense layer with 256 neurons and ReLU activation to learn more complex musical features before making predictions.

5. Add the final Dense layer with Softmax activation to predict the probability of every possible next note or chord.

6. Compile the model using the Adam optimizer and categorical cross-entropy loss, then display the model summary to verify the network architecture.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense

model = Sequential()

model.add(LSTM(
    512,
    input_shape=(network_input.shape[1], network_input.shape[2]),
    return_sequences=True
))
model.add(Dropout(0.3))

model.add(LSTM(512, return_sequences=True))
model.add(Dropout(0.3))

model.add(LSTM(512))
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.3))

model.add(Dense(network_output.shape[1], activation="softmax"))

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam"
)

model.summary()

1. Create a checkpoint that automatically saves the model whenever its training loss improves.

2. Train the deep learning model using the prepared input and output data for 5 epochs with a batch size of 64.

3. Monitor the training loss during each epoch and save only the best-performing model.

4. Store the training history, including the loss values from each epoch, for later analysis or visualization.


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="loss",
    verbose=1,
    save_best_only=True,
    mode="min"
)

history = model.fit(
    network_input,
    network_output,
    epochs=5,
    batch_size=64,
    callbacks=[checkpoint]
)

1. Load the best trained deep learning model that was saved during training.

2. Randomly select one input sequence from the training data as the starting point for music generation.

3. Create a dictionary to convert predicted integer values back into their original note or chord names.

4. Use the selected input sequence as the initial pattern and create an empty list to store the generated notes.

5. Repeat the prediction process 500 times to generate a sequence of new musical notes and chords.

6. For each prediction, convert the model's output probabilities into the most likely note or chord and save it.

7. Add the newly predicted note to the input pattern and remove the oldest note to maintain a fixed sequence length of 100.

8. Continue this process until 500 notes are generated, resulting in a complete sequence of predicted music.


In [ ]:
from tensorflow.keras.models import load_model
import random

model = load_model("best_model.keras")

start = random.randint(0, len(network_input) - 1)

int_to_note = {number: note for number, note in enumerate(pitchnames)}

pattern = network_input[start]
prediction_output = []

for _ in range(500):
    prediction_input = np.reshape(pattern, (1, len(pattern), 1))
    prediction = model.predict(prediction_input, verbose=0)

    index = np.argmax(prediction)
    result = int_to_note[index]
    prediction_output.append(result)

    pattern = np.append(pattern, index / float(len(pitchnames)))
    pattern = pattern[1:]

1. Create an empty list to store all the generated notes and chords, and initialize the starting time (offset) for the music.

2. Process each predicted note or chord generated by the deep learning model one by one.

3. Check whether the prediction represents a chord or a single musical note.

4. If it is a chord, split it into individual notes and create each note separately.

5. Combine all the individual notes into a single chord and assign it a position in the music timeline.

6. If it is a single note, create a musical note object and assign it a position in the timeline.

7. Set the instrument for every generated note or chord to Piano.

8. Increase the time offset after each note or chord so they are played one after another in sequence.

9. Combine all the generated notes and chords into a complete music stream.

10. Save the final music stream as a MIDI file named `generated_music.mid` and display a success message indicating that the music has been generated successfully.


In [ ]:

from music21 import stream

offset = 0
output_notes = []

for pattern in prediction_output:

    if "." in pattern or pattern.isdigit():
        notes_in_chord = pattern.split(".")
        chord_notes = []

        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.storedInstrument = instrument.Piano()
            chord_notes.append(new_note)

        new_chord = chord.Chord(chord_notes)
        new_chord.offset = offset
        output_notes.append(new_chord)

    else:
        new_note = note.Note(pattern)
        new_note.offset = offset
        new_note.storedInstrument = instrument.Piano()
        output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)
midi_stream.write("midi", fp="generated_music.mid")

print("Music generated successfully: generated_music.mid")



1. Initialize the Pygame library and its mixer module to enable audio playback.

2. Load the generated MIDI file into the music player and start playing it.

3. Display a message indicating that the generated music has started playing.

4. Continuously check whether the music is still playing and wait until it finishes.

5. Display a confirmation message after the music playback is completed.


In [ ]:
import pygame
import time


pygame.init()
pygame.mixer.init()
pygame.mixer.music.load("generated_music.mid")
pygame.mixer.music.play()

print("Playing generated_music.mid...")


while pygame.mixer.music.get_busy():
    time.sleep(1)

print("Finished playing.") 

In [ ]:
#pygame.mixer.music.stop()